# Set-Up

## Imports & Param Defs

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import sys
import polars as pl
from pathlib import Path
from pprint import pprint

import processing_utils as pu

# `conform_data` (date parsing, pain conversion) lives with the master-dataset ETL
sys.path.append(str(Path().resolve().parent / "extract_master_dataset"))
import conform_data as cd

path = Path().resolve().parent.parent / "data"
OUTPUT_PATH = path / "outputs"

# vital-sign ranges, derived columns, and score definitions
config = pu.load_config()

In [24]:
# list for tracking df lengths after transformations
tmp_lens = []

## Helper Functions

In [ ]:
def track_len(df: pl.DataFrame | pl.LazyFrame, tmp_lens: list):
    """Helper for tracking df len after each transformation."""
    if isinstance(df, pl.LazyFrame):
        length = df.select(pl.len()).collect().item()
    else:
        length = df.shape[0]
    print(f"Current length:  {length:,}")
    print(f"Change in Rows:  {length - tmp_lens[-1] if tmp_lens else length:,}")
    print(f"Pct of Original: {length / tmp_lens[0] * 100 if tmp_lens else 100:.2f}%")
    tmp_lens.append(length)

# Define Data

In [26]:
master_df = pl.scan_parquet(path / "outputs/master_dataset.parquet")
track_len(master_df, tmp_lens)

Current length:  448,972
Change in Rows:  448,972
Pct of Original: 100.00%


In [27]:
master_df.head().show()

subject_id,hadm_id,stay_id,intime,outtime,anchor_age,gender,anchor_year,dod,admittime,dischtime,deathtime,ethnicity,edregtime,edouttime,insurance,in_year,age,outcome_inhospital_mortality,ed_los,intime_icu,time_to_icu_transfer,outcome_icu_transfer_12h,outcome_hospitalization,outcome_critical,ethnicity_was_null,n_ed_30d,n_ed_90d,n_ed_365d,next_ed_visit_time,next_ed_visit_time_diff,outcome_ed_revisit_3d,n_hosp_30d,n_icu_30d,n_hosp_90d,n_icu_90d,n_hosp_365d,…,eci_PHTN,eci_PVD,eci_HTN1,eci_HTN2,eci_Paralysis,eci_NeuroOther,eci_Pulmonary,eci_DM1,eci_DM2,eci_Hypothyroid,eci_Renal,eci_Liver,eci_PUD,eci_HIV,eci_Lymphoma,eci_Tumor2,eci_Tumor1,eci_Rheumatic,eci_Coagulopathy,eci_Obesity,eci_WeightLoss,eci_FluidsLytes,eci_BloodLoss,eci_Anemia,eci_Alcohol,eci_Drugs,eci_Psychoses,eci_Depression,ed_temperature_last,ed_heartrate_last,ed_resprate_last,ed_o2sat_last,ed_sbp_last,ed_dbp_last,ed_pain_last,n_med,n_medrecon
i64,i64,i64,datetime[μs],datetime[μs],i64,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,i32,i64,bool,duration[μs],datetime[μs],duration[μs],bool,bool,bool,bool,i32,i32,i32,datetime[μs],duration[μs],bool,i32,i32,i32,i32,i32,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f64,f64,f64,f64,i64,i64,f64,u32,u32
19416727,21289895,35425579,2189-06-16 21:35:00,2189-06-16 22:15:39,69,"""M""",2189,null,null,null,null,"""UNKNOWN""",null,null,null,2189,69,false,40m 39s,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,37.2,122.0,28.0,100.0,108,89,NaN,8,0
10628682,22653302,35851515,2133-05-12 23:17:00,2133-05-13 04:14:00,38,"""M""",2133,null,null,null,null,"""UNKNOWN""",null,null,null,2133,38,false,4h 57m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,96.1,85.0,18.0,100.0,133,60,NaN,7,11
14125406,27740734,32795976,2181-11-18 18:42:00,2181-11-18 22:55:00,76,"""F""",2181,null,null,null,null,"""UNKNOWN""",null,null,null,2181,76,false,4h 13m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,96.6,92.0,16.0,96.0,136,82,4.0,4,4
11665270,26049594,33012761,2162-09-09 21:24:00,2162-09-10 00:58:00,76,"""F""",2162,null,null,null,null,"""UNKNOWN""",null,null,null,2162,76,false,3h 34m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,97.0,105.0,18.0,97.0,124,70,0.0,4,14
14143633,29781340,30136351,2169-10-02 05:47:00,2169-10-02 18:23:00,34,"""F""",2169,null,null,null,null,"""UNKNOWN""",null,null,null,2169,34,false,12h 36m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,98.3,55.0,18.0,99.0,136,84,0.0,4,1


# 1. General Filter - Age, triage_acuity

In [28]:
# filter out children
master_df = master_df.filter(pl.col('age') >= 18)

track_len(master_df, tmp_lens)

Current length:  448,804
Change in Rows:  -168
Pct of Original: 99.96%


In [29]:
# drop null `triage_acuity`
master_df = master_df.filter(pl.col('triage_acuity').is_not_null())

track_len(master_df, tmp_lens)

Current length:  441,437
Change in Rows:  -7,367
Pct of Original: 98.32%


# 2. Outlier Detection

In [30]:
# convert temperature columns from Fahrenheit to Celsius
master_df = pu.convert_temp_to_celsius(master_df)

In [31]:
# how many values fall in each outlier / valid band, per vital-sign column
vitals_cols = pu.vital_columns(master_df.collect_schema().names(), config['vitals_valid_range'])
pu.outlier_counts(master_df.select(vitals_cols), config['vitals_valid_range'])

variable,< outlier_low,"[outlier_low, valid_low)","[valid_low, valid_high]","(valid_high, outlier_high]",> outlier_high
str,u32,u32,u32,u32,u32
"""triage_temperature""",512,29,423329,0,10
"""triage_heartrate""",0,0,430515,0,7
"""triage_resprate""",0,0,427096,0,4
"""triage_o2sat""",0,0,426773,6,43
"""triage_sbp""",0,0,429234,0,19
…,…,…,…,…,…
"""ed_resprate_last""",0,0,425263,0,0
"""ed_o2sat_last""",0,0,414572,0,11
"""ed_sbp_last""",0,0,425531,0,1


In [32]:
# null values beyond the outlier bounds, clip values beyond the valid bounds
master_df = pu.remove_outliers(master_df, config['vitals_valid_range'])

track_len(master_df, tmp_lens)

Current length:  441,437
Change in Rows:  0
Pct of Original: 98.32%


# 3. Train / Test Split

80/20 split, seeded. Sorted first so the split is reproducible.

In [33]:
SEED = 10
TRAIN_FRAC = 0.8

master_df = master_df.sort('subject_id', 'intime', 'stay_id').collect()
train_df, test_df = pu.train_test_split(master_df, frac=TRAIN_FRAC, seed=SEED)

print(f"Training dataset size = {train_df.height:,}")
print(f"Testing dataset size  = {test_df.height:,}")

Training dataset size = 353,150
Testing dataset size  = 88,287


# 4. Missing Value Imputation

Median-impute vital signs, using medians computed on the training set only.

In [34]:
# missing values in the training set
(
    train_df.null_count().transpose(include_header=True, header_name='column', column_names=['n_missing'])
    .with_columns(pct_missing = pl.col('n_missing') / train_df.height * 100)
    .filter(pl.col('n_missing') > 0)
    .sort('n_missing', descending=True)
)

column,n_missing,pct_missing
str,u32,f64
"""deathtime""",349869,99.070933
"""dod""",339411,96.109585
"""intime_icu""",328916,93.13776
"""time_to_icu_transfer""",328916,93.13776
"""time_to_icu_transfer_hours""",328916,93.13776
…,…,…
"""triage_resprate""",11486,3.252442
"""triage_dbp""",10776,3.051395
"""triage_sbp""",9748,2.7603


In [35]:
impute_cols = pu.vital_columns(train_df.columns, config['vitals_valid_range'], exclude_types=('acuity',))
train_df, (test_df,), medians = pu.impute_median(train_df, [test_df], impute_cols)
pprint(medians)

{'ed_dbp_last': 73.0,
 'ed_heartrate_last': 77.0,
 'ed_o2sat_last': 99.0,
 'ed_pain_last': 0.0,
 'ed_resprate_last': 17.0,
 'ed_sbp_last': 126.0,
 'ed_temperature_last': 36.72222222222222,
 'triage_dbp': 77.0,
 'triage_heartrate': 84.0,
 'triage_o2sat': 99.0,
 'triage_pain': 4.0,
 'triage_resprate': 18.0,
 'triage_sbp': 133.0,
 'triage_temperature': 36.66666666666667}


# 5. Add Score Values

Score definitions (bins and weights) are in `config.json`.

In [36]:
SCORES = ['CCI', 'CART', 'REMS', 'NEWS', 'NEWS2', 'MEWS']  # also available: 'SERP2d', 'SERP7d', 'SERP30d'

train_df = pu.add_scores(train_df, config, SCORES)
test_df = pu.add_scores(test_df, config, SCORES)

# 6. Output Train and Test Data

In [37]:
train_df.write_parquet(OUTPUT_PATH / "train.parquet")
test_df.write_parquet(OUTPUT_PATH / "test.parquet")

# 7. ED Vital Sign Time Series

Resample each stay's `vitalsign` records to a regular grid, then apply the same temperature, outlier and imputation steps.

In [38]:
RESAMPLE_FREQ = '1h'
ED_VITALS = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']

ed_vitalsign = (
    cd.conform_dates(pl.scan_csv(path / "ed" / "vitalsign.csv"), ['charttime'])
    .select('subject_id', 'stay_id', 'charttime', *[pl.col(v).alias(f'ed_{v}') for v in ED_VITALS])  # drops `rhythm`
    .with_columns(pl.col('ed_pain').map_elements(cd.convert_str_to_float, return_dtype=pl.Float64))
)

In [39]:
vital_ts_cols = [f'ed_{v}' for v in ED_VITALS]

vitalsign_resampled = (
    pu.resample_vitals(ed_vitalsign, vital_ts_cols, freq=RESAMPLE_FREQ)
    .pipe(pu.convert_temp_to_celsius)
    .pipe(pu.remove_outliers, config['vitals_valid_range'])
    .collect()
)
vitalsign_resampled, _, _ = pu.impute_median(vitalsign_resampled, [], vital_ts_cols)
vitalsign_resampled.head()

subject_id,stay_id,charttime,ed_temperature,ed_heartrate,ed_resprate,ed_o2sat,ed_sbp,ed_dbp,ed_pain
i64,i64,datetime[μs],f64,f64,f64,f64,f64,f64,f64
11714491,30000012,2126-02-14 20:22:00,37.111111,96.0,18.0,93.0,160.0,54.0,0.0
11714491,30000012,2126-02-14 21:22:00,37.086235,91.223881,16.507463,94.791045,145.671642,51.014925,0.0
11714491,30000012,2126-02-14 22:22:00,37.06136,86.447761,15.014925,96.58209,131.343284,48.029851,0.0
11714491,30000012,2126-02-14 23:22:00,37.036484,81.671642,13.522388,98.373134,117.014925,45.044776,0.0
11714491,30000012,2126-02-15 00:22:00,37.011609,84.656716,14.746269,99.58209,125.38806,48.074627,0.0


In [40]:
vitalsign_resampled.write_parquet(OUTPUT_PATH / f"ed_vitalsign_{RESAMPLE_FREQ}_resampled.parquet")